# ARXIV PROJECT REPORT
- **Course:** CSC14119 - Introduction to Data Science
- **Group:** 03
- **Project:** arXiv Multi-Version LaTeX Source Dataset & Performance Evaluation
- **Date:** October 2026
- **Report Site:** [GitHub Pages Documentation](https://duynhat1705.github.io/Intro2DS-Group3-Report/)
- **Repository:** [Report Repository](https://github.com/DuyNhat1705/Intro2DS-Group3-Report) | [Shared Code Repository](https://github.com/DuyNhat1705/Intro2DS-Group3-Source)
---

# Setup

This section sets up the execution environment, imports required libraries, configures file system paths, and defines data loader functions. 

All tables and figures in this notebook are executed and rendered dynamically from the pipeline's per-paper JSONL log files and auditable system benchmark metrics.

In [1]:
# 0.1 Imports and Pipeline Configuration
import os
import sys
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from IPython.display import display, HTML
except ImportError:
    display = print

# Project Milestone 1 — Data Crawling
# 1. Overview

## 1.1 Goal and Scope
The goal of Milestone 1 is to build an automated, auditable, and resilient data engineering pipeline capable of collecting, curating, and evaluating a dataset of scientific research papers from **arXiv** and citation metadata from **Semantic Scholar**.

The scope encompasses:
1. **Multi-Version LaTeX Source Ingestion:** Downloading every revision ($v_1, v_2, \dots, v_n$) of each target paper from `export.arxiv.org/src/<id>v<n>`.
2. **Strict Content Extraction:** Unpacking compressed source archives, retaining `.tex` text files and `.bib` bibliography files, and discarding compiled binaries.
3. **Automated Figure Removal:** Stripping raster and vector figures (`.png`, `.jpg`, `.pdf`, `.eps`, `.ps`) to isolate semantic LaTeX code and drastically reduce disk footprint.
4. **Bibliographic Reference Scraping:** Querying the Semantic Scholar Academic Graph API to obtain structured metadata for all cited references.
5. **Rigorous Auditing on Google Colab:** Benchmarking running times, peak/average RAM consumption, and disk footprint on a CPU-only testbed.

---

## 1.2 Output Structure per Paper
Every harvested paper is stored in a clean, self-contained directory tree under the `data/` directory:

```
data/
└── <arxiv_id>/
    ├── metadata.json                 # Paper metadata (title, authors, categories, versions, abstract)
    ├── references.json               # Extracted references and Semantic Scholar metadata
    ├── v1/                           # Version 1 LaTeX source
    │   ├── main.tex
    │   ├── sections/
    │   │   ├── intro.tex
    │   │   └── methods.tex
    │   └── references.bib
    ├── v2/                           # Version 2 LaTeX source
    │   ├── main.tex
    │   └── references.bib
    └── vn/                           # Version n LaTeX source (latest version)
```

---

## 1.3 Pipeline Architecture Diagram

# 2. Implementation
## 2.1 Entry Discovery

---

## 2.2 Source Download

---

## 2.3 Extraction & Figure Removal

---

## 2.4 Reference Metadata Scraping

---

## 2.5 Robustness, Resumability & Logging
- **Resumable State Management:** The scraper writes its progress to an atomic `state.json` file. Processed papers are indexed, enabling the scraper to resume instantly without re-downloading existing records.
- **Per-Paper JSONL Logging:** Emits structured logs containing per-stage latencies, byte counts, version numbers, and HTTP status codes into `logs/scraping_summary.jsonl`.
- **Categorization of Failures:** Errors are trapped into explicit taxonomies:
  - `NETWORK_TIMEOUT`: Socket read/connect timeout.
  - `RATE_LIMIT_EXCEEDED`: HTTP 429 encountered past maximum retry budget.
  - `PDF_ONLY_VERSION`: Paper source contains no LaTeX code.
  - `CORRUPT_ARCHIVE`: Failed gzip/tar checksum or corrupted tar headers.
  - `SEMANTIC_SCHOLAR_NOT_FOUND`: Reference metadata unavailable on S2.

---

## 2.6 Tools and Libraries Table
## Third-Party Libraries

**Data collection**

| Library | Used in | Purpose |
|:--|:--|:--|
| `arxiv` 4.0.1 | `get_metadata.py` | arXiv API wrapper for metadata and paper discovery (page size 100, ≥3 s delay, batch queries via `get_entry_discovery_batch`) |
| `requests` 2.34.0 | `get_full_source.py`, `get_extracting_reference.py` | Downloads LaTeX sources from `export.arxiv.org/src/{id}`; queries Semantic Scholar with API key and exponential backoff on 429/5xx |

**Data processing and storage**

| Library | Used in | Purpose |
|:--|:--|:--|
| `pandas` 3.1.0 | `scraper.ipynb` | Builds `papers`, `references`, `references_all` DataFrames; schema checks, deduplication, summary stats |
| `pyarrow` 26.0.0 | `scraper.ipynb` | Parquet backend: compact storage, strict types, fast columnar reads |

**Monitoring and visualization**

| Library | Used in | Purpose |
|:--|:--|:--|
| `psutil` 7.2.2 | `scraper.ipynb` | Background telemetry: RSS memory, CPU, disk usage |
| `tqdm` 4.70.1 | `scraper.ipynb` | Progress bars, rates, ETA for crawl loops |
| `matplotlib` 3.11.2 | `scraper.ipynb` | Sanity-check plots: memory curves, latency histograms, storage savings |
| `tracemalloc` (stdlib) | `get_full_source.py` | Peak RAM tracking via `measure()` context manager |

**Archive handling (stdlib)**

| Module | Used in | Purpose |
|:--|:--|:--|
| `tarfile` | `get_full_source.py` | Streaming extraction of `.tex`/`.bib` from `.tar(.gz)` with path-traversal protection |
| `gzip` | `get_full_source.py` | Decompresses single-file `.gz` submissions to `main.tex` (magic-byte check) |
| `zipfile` | `get_full_source.py` | Extracts `.tex`/`.bib` from ZIPs; filters out Word (`docx`) files |

**Filesystem and persistence (stdlib)**

| Module | Used in | Purpose |
|:--|:--|:--|
| `pathlib` | `utils.py`, `src/*.py` | Cross-platform paths (`data/<yymm-nnnnn>/v1/`), repo root resolution |
| `shutil` / `os` | `get_full_source.py`, `src/*.py` | Atomic writes (`.tmp` + `os.replace`), cleanup, true disk-cluster size (`GetDiskFreeSpaceW` / `statvfs`) |
| `json` | `src/*.py`, `scraper.ipynb` | Persists `metadata.json`, `references.json`, `source_state.json`, `summary_stats.json` |
| `csv` | `scraper.ipynb` | Exports report tables to `data/report_exports/` for Jupyter Book |

**Utilities and concurrency (stdlib)**

| Module | Used in | Purpose |
|:--|:--|:--|
| `concurrent.futures` / `threading` | `scraper.ipynb` | Runs the arXiv lane (Phase A+B, 1 connection), Semantic Scholar lane (Phase C), and monitor thread in parallel |
| `re` | `src/*.py`, `scraper.ipynb` | ID normalization, version parsing (`v(\d+)$`), ISO-8601 validation |
| `logging` | `src/*.py`, `scraper.ipynb` | Timestamped logs to stderr and `scraper.log` (429s, retries, batch progress) |
| `time` | `src/*.py`, `scraper.ipynb` | Throttling, backoff, `perf_counter` / `monotonic` timing |

**Environment and Tooling**

| Tool | Version | Purpose |
|:--|:--|:--|
| Python | ≥ 3.9 (rec. 3.12, tested 3.14) | Core runtime (pandas 3.x needs ≥ 3.11) |
| Jupyter / `ipykernel` | 7.4.0 | Hosts `scraper.ipynb` orchestration and EDA |
| Google Colab + Drive | CPU runtime | Benchmark testbed for 25,000 papers; state persisted in `MyDrive/Intro2DS-Group3/data` |
| Conda / Mamba | `intro2ds-env.yml` | Reproducible env (Python 3.12, pinned pip packages) |
| Git / GitHub | n/a | Branches `scraper`, `main`, `get-extract-ref`; clone inside Colab |
| Jupyter Book | n/a | Builds the Milestone 1 report from `data/report_exports/` |

**External Services**

| Service | Endpoint | Purpose |
|:--|:--|:--|
| arXiv API / export | Atom API, `export.arxiv.org/src/{id}` | Metadata and versioned LaTeX sources (≥ 3 s delay, single connection) |
| Semantic Scholar Graph API | `/graph/v1/paper/arXiv:{id}/references`, `/paper/batch` | Citation graph: reference IDs, titles, authors, dates |


# 3. Data Statistics

All statistics and visualizations in this section are dynamically computed by executing code cells directly against the scraped dataset logs.

## 3.1 Scraping Success Rate & Failure Breakdown
We assess the reliability of our scraping pipeline across all attempted papers, evaluating both the overall success rate and specific categories of failure.

## 3.2 Paper Size Analysis: Before vs. After Figure Removal
Stripping figure assets (PNG, JPG, EPS, standalone figure PDFs) drastically reduces disk footprint while preserving all essential LaTeX code and BibTeX entries.

## 3.3 Reference Statistics & Metadata Success Rate
We examine the distribution of bibliographic references per paper and the success rate of retrieving structured reference metadata via Semantic Scholar.

## 3.4 Additional Breakdowns: Versions, Categories & Timeline
Exploring dataset diversity across paper revision counts, scientific subfields, and publication years.

# 4. Performance Evaluation

## 4.1 Benchmarking Methodology & Testbed Environment
To ensure reproducible benchmarking, the complete scraping and extraction pipeline was evaluated on a standard **Google Colab CPU-only** virtual machine instance.

### Auditing Tools & Instrumentation
- **Testbed Hardware Specifications:** Inspected via `lscpu`, `free -h`, and `df -h`.
- **Peak RAM Measurement:** Audited via GNU time (`/usr/bin/time -v`) and process monitoring with `psutil.Process().memory_info().rss`.
- **Average RAM Consumption:** Monitored via a background daemon thread periodically sampling resident set size (RSS) every 500 ms throughout execution.
- **Disk Utilization:** Audited using `du -sm` and `shutil.disk_usage` at successive pipeline milestones.
- **End-to-End Wall Time:** Measured across all decoupled components from initial entry discovery to final structured storage.

## 4.2 Running Time Analysis
We analyze the runtime of each pipeline stage: entry discovery, source download, archive extraction, figure stripping, and reference scraping.

## 4.3 Memory Footprint & Storage Analysis
Tracking resident memory consumption over time and monitoring disk allocation before and after archive processing.

## 4.4 Performance Metrics Summary Table
This table compiles all required benchmark metrics specified in `Report_Requirement.md`.

## 4.5 Bottleneck Discussion: Rate Limiting vs. Compute & IO
- **Dominant Bottleneck (Network Rate Limiting):** The overriding bottleneck in our pipeline is the mandatory **3.0-second delay per request** enforced by arXiv's access policies, along with Semantic Scholar's rate limits. CPU computation (regex extraction, unzipping) and local disk IO together consume less than 15% of total elapsed runtime.
- **Peak Disk vs. Final Storage:** During archive extraction, intermediate compressed tar files temporarily inflate disk usage before figure removal and cleanup. With immediate per-paper cleanup, peak disk usage remains strictly bounded, comfortably within Google Colab's ~108 GB allocation.
- **Parallelization Trade-offs:** While reference scraping from Semantic Scholar or extracting local `.tex` archives can be parallelized using multi-threading or `asyncio`, arXiv source download cannot be aggressively parallelized without violating courtesy limits and incurring IP blocks.

# 5. Conclusion

## 5.1 Key Results & Findings
- **Reliable Automated Ingestion:** Successfully constructed an end-to-end automated harvester for arXiv multi-version LaTeX sources and Semantic Scholar citation metadata.
- **Storage Optimization:** Figure stripping achieved an average size reduction exceeding 90%, enabling dense, cost-effective storage suitable for Colab's environment.
- **Resilient Execution:** State checkpointing and structured error categorization ensure the pipeline resumes seamlessly from transient network failures without duplicating downloads.

---

## 5.2 Limitations & Challenges
- **Rate-Limiting Bottleneck:** The compulsory 3.0-second pacing on `export.arxiv.org` limits single-IP throughput to ~1,200 requests per hour.
- **Non-Standard Submissions:** A small fraction of submissions do not provide LaTeX sources (e.g., direct PDF submissions or MS Word conversions).
- **Semantic Scholar Coverage:** Newer or highly specialized preprints occasionally lack indexed citations on Semantic Scholar, requiring fallback to local BibTeX parsing.

---

## 5.3 Roadmap & Plan for Milestone 2
- **LaTeX AST Cleaning & Parsing:** Parse `.tex` abstract syntax trees to tokenize equations, section headers, and prose text.
- **Citation Graph Construction:** Build and analyze citation networks from the scraped bibliographic metadata.
- **Tabular & Multimodal Preprocessing:** Extract embedded LaTeX tables and structural elements into analytical DataFrames.
- **Optimized Columnar Storage:** Transition dataset persistence into columnar formats (Parquet / Arrow / SQLite) for fast analytical querying.